In [ ]:
from autogluon.tabular import TabularPredictor

from sklearn.metrics import accuracy_score, roc_auc_score, matthews_corrcoef, f1_score, \
    precision_score, recall_score, top_k_accuracy_score, \
    confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np
import os
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="white")
sns.set_context("talk")
plt.rcParams['font.family'] = 'Arial'

## 1. Evaluate on test data for 4 class
- `sp_mode`: `s` (only use substrate concentration)
- `feat`: `tsfresh_raw` (use both tsfresh features and raw data)
- `ag_train_quality`: `best_quality` (use `best_quality` present in Autogluon training)
- `hours`: `3` (train for 3 hours)

In [ ]:
class_list = [
    'S1',
    'S2',
    'S3',
    'S4',
    # 'S5',
]
class_num = len(class_list)

ag_model_root = './AutogluonModels/'
sp_mode = 's'
feat = 'tsfresh_raw'
ag_train_quality = 'best_quality'
hours = 3.0
label_col = 'class'
random_seed = 42

# load test data
test_tsfresh_data = pd.read_csv(os.path.join(f'data/', f'{str(class_num)}_class', 'test_tsfresh_feat.csv'))
test_raw_data = pd.read_csv(os.path.join(f'data/', f'{str(class_num)}_class', 'test.csv'))
if sp_mode == 's':
    test_tsfresh_data.drop(columns=[c for c in test_tsfresh_data.columns if c.startswith('p')], inplace=True)
    test_raw_data.drop(columns=[c for c in test_raw_data.columns if c.startswith('p')], inplace=True)
elif sp_mode == 'p':
    test_tsfresh_data.drop(columns=[c for c in test_tsfresh_data.columns if c.startswith('s')], inplace=True)
    test_raw_data.drop(columns=[c for c in test_raw_data.columns if c.startswith('s')], inplace=True)
if feat == 'tsfresh':
    test_merge_data = test_tsfresh_data
elif feat == 'raw':
    test_merge_data = test_raw_data
elif feat == 'tsfresh_raw':
    test_raw_data.drop(columns=['class'], inplace=True)
    test_merge_data = pd.merge(test_tsfresh_data, test_raw_data, on='id')
test_merge_data.replace([np.inf, -np.inf], np.nan, inplace=True)
y_test = test_merge_data[label_col]
test_merge_data_nolabel = test_merge_data.drop(columns=[label_col])  # delete label column

In [ ]:
# load model
model_root = os.path.join(ag_model_root, f'{str(class_num)}_class', f'{feat}_feat__{sp_mode}__{ag_train_quality}__{hours}h')
predictor = TabularPredictor.load(model_root)
predictor.leaderboard()

In [ ]:
# predict on test data
y_pred = predictor.predict(test_merge_data_nolabel)
y_proba = predictor.predict_proba(test_merge_data_nolabel)

In [ ]:
# show performance
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'ROC_AUC: {roc_auc_score(y_test, y_proba, multi_class="ovr"):.4f}')
print(f'MCC: {matthews_corrcoef(y_test, y_pred):.4f}')
print(f'F1: {f1_score(y_test, y_pred, average="weighted"):.4f}')
print(f'Precision: {precision_score(y_test, y_pred, average="weighted"):.4f}')
print(f'Recall: {recall_score(y_test, y_pred, average="weighted"):.4f}')
print(f'Top-2 accuracy: {top_k_accuracy_score(y_test, y_proba, k=2):.4f}')
print(f'Top-3 accuracy: {top_k_accuracy_score(y_test, y_proba, k=3):.4f}')
print('================================')
print(f'Classification report:\n{classification_report(y_test, y_pred)}')

In [ ]:
# show confusion matrix
plt.figure(figsize=(4, 4), dpi=300)
# plt.rcParams.update({'font.size': 20})
cm = confusion_matrix(y_test, y_pred, labels=class_list)
sns.heatmap(cm, annot=True, fmt='g', cmap=plt.cm.Blues, cbar=False)
tick_marks = np.arange(len(class_list)) + 0.5
if class_num == 5:
    class_list = ['S1', 'S2', 'S3', 'S4', 'S5']
plt.xticks(tick_marks, class_list, rotation=0, fontweight='bold')
plt.yticks(tick_marks, class_list, rotation=0, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('True')
# plt.title(f'Confusion Matrix of Prediction on Test Data', fontsize=17)
plt.show()

# plt.savefig(f'./AutogluonModels/{str(class_num)}_class/{feat}_feat__{sp_mode}__{ag_train_quality}__{hours}h/confusion_matrix.png', bbox_inches='tight')
# print(f'Confusion matrix saved at ./AutogluonModels/{str(class_num)}_class/{feat}_feat__{sp_mode}__{ag_train_quality}__{hours}h/confusion_matrix.png')

In [ ]:
# show chord diagram
import openchord as ocd

plt.figure(figsize=(6, 6), dpi=300)
fig = ocd.Chord(cm, class_list, radius=100)
fig.font_size = 28

fig.colormap = ['#FFA878', '#79A1E3', '#88D59A', '#E37275']

fig.show()

In [ ]:
y_proba = y_proba[class_list]
y_proba['class'] = test_merge_data['class']

def plot_scatter_figs(data, ax):
    sns.scatterplot(data=data['S1'], ax=ax, legend=False, marker='.', edgecolor='none', color='#FFA878')
    sns.scatterplot(data=data['S2'], ax=ax, legend=False, marker='.', edgecolor='none', color='#79A1E3')
    sns.scatterplot(data=data['S3'], ax=ax, legend=False, marker='.', edgecolor='none', color='#88D59A')
    sns.scatterplot(data=data['S4'], ax=ax, legend=False, marker='.', edgecolor='none', color='#E37275')

# 4 figs
fig, axes = plt.subplots(2, 2, figsize=(7, 6.5), dpi=300)
axes = axes.flatten() 
from matplotlib.lines import Line2D

# S1
df_S1 = y_proba[y_proba['class'] == 'S1']
df_S1 = df_S1.sort_values(by=['S1', 'S2', 'S3', 'S4'], ascending=False).reset_index(drop=True)
plot_scatter_figs(df_S1, axes[0])
axes[0].set_title('S1', fontweight='bold')

# S2
df_S2 = y_proba[y_proba['class'] == 'S2']
df_S2 = df_S2.sort_values(by=['S2', 'S3', 'S4', 'S1'], ascending=False).reset_index(drop=True)
plot_scatter_figs(df_S2, axes[1])
axes[1].set_title('S2', fontweight='bold')

# S3
df_S3 = y_proba[y_proba['class'] == 'S3']
df_S3 = df_S3.sort_values(by=['S3', 'S4', 'S1', 'S2'], ascending=False).reset_index(drop=True)
plot_scatter_figs(df_S3, axes[2])
axes[2].set_title('S3', fontweight='bold')

# S4
df_S4 = y_proba[y_proba['class'] == 'S4']
df_S4 = df_S4.sort_values(by=['S4', 'S1', 'S2', 'S3'], ascending=False).reset_index(drop=True)
plot_scatter_figs(df_S4, axes[3])
axes[3].set_title('S4', fontweight='bold')

custom_handles = [
    Line2D([0], [0], color='#FFA878', marker='.', markersize=15, linestyle='none', label='S1'),
    Line2D([0], [0], color='#79A1E3', marker='.', markersize=15, linestyle='none', label='S2'),
    Line2D([0], [0], color='#88D59A', marker='.', markersize=15, linestyle='none', label='S3'),
    Line2D([0], [0], color='#E37275', marker='.', markersize=15, linestyle='none', label='S4'),
]
fig.legend(
    handles=custom_handles,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.03),
    ncols=4,
    # fontsize=12,
    frameon=False
)

# plt.subplots_adjust(hspace=0.4, wspace=0.3, bottom=0.2)
# ylim
for ax in axes[:]:
    ax.set_ylim(-0.05, 1.05)

for ax_idx, ax in enumerate(axes):
    ax.set_ylabel('') # Clear all y-labels first
    # Bold xticks and yticks
    for tick_label in ax.get_xticklabels():
        tick_label.set_fontweight('bold')
    for tick_label in ax.get_yticklabels():
        tick_label.set_fontweight('bold')
    for spine in ax.spines.values(): # ax.spines is a dictionary containing 'top', 'bottom', 'left', 'right'
        spine.set_linewidth(1) # Set line width to 1
    
    ax.tick_params(
        axis='both',
        which='major',
        direction='out',
        length=5,
        width=1,
        color='black',
        pad=5,
        bottom=True,  # Ensure ticks are drawn on the required edges
        left=True,    # Ensure ticks are drawn on the required edges
        top=False,    # Typically do not show ticks on the top and right
        right=False
    )

    if ax_idx % 2 == 1:
        ax.set_yticklabels([])
    else:
        ax.set_ylabel('Probability')
    if ax_idx < 2:
        ax.set_xticklabels([])
    else:
        ax.set_xlabel('Kinetic sample number')

## 2. Evaluate on test data for 5 class
- `sp_mode`: `s` (only use substrate concentration)
- `feat`: `tsfresh_raw` (use both tsfresh features and raw data)
- `ag_train_quality`: `best_quality` (use `best_quality` present in Autogluon training)
- `hours`: `3` (train for 3 hours)

In [ ]:
class_list = [
    'S1',
    'S2',
    'S3',
    'S4',
    'S5',
]
class_num = len(class_list)

ag_model_root = './AutogluonModels/'
sp_mode = 's'
feat = 'tsfresh_raw'
ag_train_quality = 'best_quality'
hours = 3.0
label_col = 'class'
random_seed = 42

# load test data
test_tsfresh_data = pd.read_csv(os.path.join(f'data/', f'{str(class_num)}_class', 'test_tsfresh_feat.csv'))
test_raw_data = pd.read_csv(os.path.join(f'data/', f'{str(class_num)}_class', 'test.csv'))
if sp_mode == 's':
    test_tsfresh_data.drop(columns=[c for c in test_tsfresh_data.columns if c.startswith('p')], inplace=True)
    test_raw_data.drop(columns=[c for c in test_raw_data.columns if c.startswith('p')], inplace=True)
elif sp_mode == 'p':
    test_tsfresh_data.drop(columns=[c for c in test_tsfresh_data.columns if c.startswith('s')], inplace=True)
    test_raw_data.drop(columns=[c for c in test_raw_data.columns if c.startswith('s')], inplace=True)
if feat == 'tsfresh':
    test_merge_data = test_tsfresh_data
elif feat == 'raw':
    test_merge_data = test_raw_data
elif feat == 'tsfresh_raw':
    test_raw_data.drop(columns=['class'], inplace=True)
    test_merge_data = pd.merge(test_tsfresh_data, test_raw_data, on='id')
test_merge_data.replace([np.inf, -np.inf], np.nan, inplace=True)
y_test = test_merge_data[label_col]
test_merge_data_nolabel = test_merge_data.drop(columns=[label_col])  # delete label column

In [ ]:
# load model
model_root = os.path.join(ag_model_root, f'{str(class_num)}_class', f'{feat}_feat__{sp_mode}__{ag_train_quality}__{hours}h')
predictor = TabularPredictor.load(model_root)
predictor.leaderboard()

In [ ]:
# predict on test data
y_pred = predictor.predict(test_merge_data_nolabel)
y_proba = predictor.predict_proba(test_merge_data_nolabel)

In [ ]:
# show performance
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'ROC_AUC: {roc_auc_score(y_test, y_proba, multi_class="ovr"):.4f}')
print(f'MCC: {matthews_corrcoef(y_test, y_pred):.4f}')
print(f'F1: {f1_score(y_test, y_pred, average="weighted"):.4f}')
print(f'Precision: {precision_score(y_test, y_pred, average="weighted"):.4f}')
print(f'Recall: {recall_score(y_test, y_pred, average="weighted"):.4f}')
print(f'Top-2 accuracy: {top_k_accuracy_score(y_test, y_proba, k=2):.4f}')
print(f'Top-3 accuracy: {top_k_accuracy_score(y_test, y_proba, k=3):.4f}')
print('================================')
print(f'Classification report:\n{classification_report(y_test, y_pred)}')

In [ ]:
# show confusion matrix
plt.figure(figsize=(5, 5), dpi=300)
# plt.rcParams.update({'font.size': 20})
cm = confusion_matrix(y_test, y_pred, labels=class_list)
sns.heatmap(cm, annot=True, fmt='g', cmap=plt.cm.Blues, cbar=False)
tick_marks = np.arange(len(class_list)) + 0.5
if class_num == 5:
    class_list = ['S1', 'S2', 'S3', 'S4', 'S5']
plt.xticks(tick_marks, ['S1', 'S2', 'S3', 'S4', 'S5'], rotation=0, fontweight='bold')
plt.yticks(tick_marks, ['S1', 'S2', 'S3', 'S4', 'S5'], rotation=0, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('True')
# plt.title(f'Confusion Matrix of Prediction on Test Data', fontsize=17)
plt.show()

# plt.savefig(f'./AutogluonModels/{str(class_num)}_class/{feat}_feat__{sp_mode}__{ag_train_quality}__{hours}h/confusion_matrix.png', bbox_inches='tight')
# print(f'Confusion matrix saved at ./AutogluonModels/{str(class_num)}_class/{feat}_feat__{sp_mode}__{ag_train_quality}__{hours}h/confusion_matrix.png')

In [ ]:
# show chord diagram
import openchord as ocd

fig = ocd.Chord(cm, class_list, radius=100)
fig.font_size = 20

fig.colormap = ['#FFA878', '#79A1E3', '#88D59A', '#E37275', '#AB9FD6']

fig.show()

In [ ]:
y_proba = y_proba[['S1', 'S2', 'S3', 'S4', 'S5']]
y_proba['class'] = test_merge_data['class']

def plot_scatter_figs(data, ax):
    sns.scatterplot(data=data['S1'], ax=ax, legend=False, marker='.', edgecolor='none', color='#FFA878')
    sns.scatterplot(data=data['S2'], ax=ax, legend=False, marker='.', edgecolor='none', color='#79A1E3')
    sns.scatterplot(data=data['S3'], ax=ax, legend=False, marker='.', edgecolor='none', color='#88D59A')
    sns.scatterplot(data=data['S4'], ax=ax, legend=False, marker='.', edgecolor='none', color='#E37275')
    sns.scatterplot(data=data['S5'], ax=ax, legend=False, marker='.', edgecolor='none', color='#AB9FD6')

# 6 figs
fig, axes = plt.subplots(2, 3, figsize=(10, 6.5), dpi=300)
axes = axes.flatten()
from matplotlib.lines import Line2D

# S1
df_S1 = y_proba[y_proba['class'] == 'S1']
df_S1 = df_S1.sort_values(by=['S1', 'S2', 'S3', 'S4', 'S5'], ascending=False).reset_index(drop=True)
plot_scatter_figs(df_S1, axes[0])
axes[0].set_title('S1', fontweight='bold')

# S2
df_S2 = y_proba[y_proba['class'] == 'S2']
df_S2 = df_S2.sort_values(by=['S2', 'S3', 'S4', 'S5', 'S1', ], ascending=False).reset_index(drop=True)
plot_scatter_figs(df_S2, axes[1])
axes[1].set_title('S2', fontweight='bold')

# S3
df_S3 = y_proba[y_proba['class'] == 'S3']
df_S3 = df_S3.sort_values(by=['S3', 'S4', 'S5', 'S1', 'S2', ], ascending=False).reset_index(drop=True)
plot_scatter_figs(df_S3, axes[2])
axes[3].set_title('S3', fontweight='bold')

# S4
df_S4 = y_proba[y_proba['class'] == 'S4']
df_S4 = df_S4.sort_values(by=['S4', 'S1', 'S2', 'S5', 'S3', ], ascending=False).reset_index(drop=True)
plot_scatter_figs(df_S4, axes[3])
axes[4].set_title('S4', fontweight='bold')

# S5
df_S5 = y_proba[y_proba['class'] == 'S5']
df_S5 = df_S5.sort_values(by=['S5', 'S4', 'S1', 'S2', 'S3', ], ascending=False).reset_index(drop=True)
plot_scatter_figs(df_S5, axes[4])
axes[2].set_title('S5', fontweight='bold')

axes[-1].axis('off')
custom_handles = [
    Line2D([0], [0], color='#FFA878', marker='.', markersize=15, linestyle='none', label='S1'),
    Line2D([0], [0], color='#79A1E3', marker='.', markersize=15, linestyle='none', label='S2'),
    Line2D([0], [0], color='#88D59A', marker='.', markersize=15, linestyle='none', label='S3'),
    Line2D([0], [0], color='#E37275', marker='.', markersize=15, linestyle='none', label='S4'),
    Line2D([0], [0], color='#AB9FD6', marker='.', markersize=15, linestyle='none', label='S5'),
]
fig.legend(
    handles=custom_handles,
    loc='center',
    bbox_to_anchor=(0.72, 0.22, 0.1, 0.1),
    # fontsize=15,
    frameon=False
)

# plt.subplots_adjust(hspace=0.4, wspace=0.3)
# ylim
for ax in axes:
    ax.set_ylim(-0.05, 1.05)
for ax_idx, ax in enumerate(axes):
    ax.set_ylabel('') # Clear all y-labels first
    # Bold the xticks and yticks
    for tick_label in ax.get_xticklabels():
        tick_label.set_fontweight('bold')
    for tick_label in ax.get_yticklabels():
        tick_label.set_fontweight('bold')
    # Set the border width to 1
    for spine in ax.spines.values(): # ax.spines is a dictionary containing 'top', 'bottom', 'left', 'right'
        spine.set_linewidth(1) # Set the line width to 1

    ax.tick_params(
        axis='both',
        which='major',
        direction='out',
        length=5,
        width=1,
        color='black',
        pad=5,
        bottom=True,  # Ensure ticks are drawn on the required edges
        left=True,    # Ensure ticks are drawn on the required edges
        top=False,    # Typically do not show ticks on the top and right
        right=False
    )
    if ax_idx % 3 != 0:
        ax.set_yticklabels([])
    else:
        ax.set_ylabel('Probability')
    if ax_idx < 2:
        ax.set_xticklabels([])
    else:
        ax.set_xlabel('Kinetic sample number')

## 3. Benchmark with data size

### 3.1 using 4_class
- 4_class
- sp_mode: `s`
- feat: `tsfresh_raw`
- ag_train_quality: `medium_quality`
- hours: `0.2`

It may take ~40 minutes, or you can skip training and use our results in `data_efficiency_test_result.csv`

In [ ]:
# You can skip this cell and use our results in `data_efficiency_test_result.csv`

class_list = [
    'S1',
    'S2',
    'S3',
    'S4',
]
class_num = len(class_list)

ag_model_root = './AutogluonModels/'
sp_mode = 's'
feat = 'tsfresh_raw'
ag_train_quality = 'medium_quality'
hours = 0.2
label_col = 'class'
random_seed = 42

train_val_tsfresh_data = pd.read_csv(os.path.join(f'data/', f'{str(class_num)}_class', 'train_val_tsfresh_feat.csv'))
train_val_raw_data = pd.read_csv(os.path.join(f'data/', f'{str(class_num)}_class', 'train_val.csv'))
test_tsfresh_data = pd.read_csv(os.path.join(f'data/', f'{str(class_num)}_class', 'test_tsfresh_feat.csv'))
test_raw_data = pd.read_csv(os.path.join(f'data/', f'{str(class_num)}_class', 'test.csv'))
train_val_raw_data.drop(columns=['class'], inplace=True)
test_raw_data.drop(columns=['class'], inplace=True)
train_val_tsfresh_data.drop(columns=[c for c in train_val_tsfresh_data.columns if c.startswith('p')], inplace=True)
train_val_raw_data.drop(columns=[c for c in train_val_raw_data.columns if c.startswith('p')], inplace=True)
test_tsfresh_data.drop(columns=[c for c in test_tsfresh_data.columns if c.startswith('p')], inplace=True)
test_raw_data.drop(columns=[c for c in test_raw_data.columns if c.startswith('p')], inplace=True)

train_val_merge_data = pd.merge(train_val_tsfresh_data, train_val_raw_data, on='id')
test_merge_data = pd.merge(test_tsfresh_data, test_raw_data, on='id')

train_val_merge_data = train_val_merge_data.sample(frac=1, random_state=random_seed).reset_index(drop=True)
test_merge_data = test_merge_data.sample(frac=1, random_state=random_seed).reset_index(drop=True)
train_val_merge_data.replace([np.inf, -np.inf], np.nan, inplace=True)
test_merge_data.replace([np.inf, -np.inf], np.nan, inplace=True)

train_val_merge_data.head()

In [ ]:
# You can skip this cell and use our results in `data_efficiency_test_result.csv`

import warnings

warnings.filterwarnings("ignore", category=UserWarning)

all_data = pd.concat([train_val_merge_data, test_merge_data], ignore_index=True)
data_size_list = np.array([500, 1000, 1500, 2000, 3000, 4000, 5000, 6000, 8000, 10000])
print(data_size_list)
test_result_df = pd.DataFrame(columns=['Data_Size', 'Accuracy', 'MCC'])
repeat_num = 5
pbar = tqdm(total=len(data_size_list) * repeat_num)
for data_size in data_size_list:
    for repeat in range(repeat_num):
        try:
            sample_data = train_test_split(all_data, train_size=data_size, shuffle=True, stratify=all_data['old_class'], )[0]
        except:
            sample_data = train_test_split(all_data, train_size=data_size, shuffle=True, stratify=all_data['class'], )[0]
        try:
            train_val_merge_data_, test_merge_data_ = train_test_split(sample_data, test_size=0.1, shuffle=True, stratify=sample_data['old_class'], )
        except:
            train_val_merge_data_, test_merge_data_ = train_test_split(sample_data, test_size=0.1, shuffle=True, stratify=sample_data['class'], )
        # print(train_val_merge_data_.shape, test_merge_data_.shape)

        metric = 'accuracy'

        predictor = TabularPredictor(
            problem_type='multiclass',
            label=label_col,
            learner_kwargs={'ignored_columns': ['id', 'old_class']},
            eval_metric=metric,
            path=f'./AutogluonModels/{str(class_num)}_class/benchmark_data_size__{data_size}__{repeat}/{feat}_feat__{sp_mode}__{ag_train_quality}__{hours}h',
            verbosity=0,
        ).fit(
            train_data=train_val_merge_data_,
            num_gpus=1,
            num_cpus=16,
            presets=ag_train_quality,
            time_limit=120,
        )
        test_result = predictor.evaluate(test_merge_data_)
        test_result_df.loc[len(test_result_df)] = [data_size, test_result['accuracy'], test_result['mcc']]

        pbar.update(1)
    
test_result_df.to_csv(f'./AutogluonModels/{str(class_num)}_class/data_efficiency_test_result.csv', index=False)

In [ ]:
# You can use our results in `data_efficiency_test_result.csv`

class_list = [
    'S1',
    'S2',
    'S3',
    'S4',
]
class_num = len(class_list)

test_result_df = pd.read_csv(f'./AutogluonModels/{str(class_num)}_class/data_efficiency_test_result.csv')
print(f'load {str(class_num)}_class data_efficiency_test_result.csv')
test_result_df

In [ ]:
plt.figure(figsize=(4.5, 4), dpi=300)
sns.lineplot(data=test_result_df, x='Data_Size', y='Accuracy', marker='o', 
             errorbar=('ci', 95),
             err_style='band',)
sns.lineplot(data=test_result_df, x='Data_Size', y='MCC', marker='o', 
             errorbar=('ci', 95),
             err_style='band',)

plt.xscale('log')
plt.xlabel('Data Size')
plt.legend(['Accuracy', 'Accuracy CI 95%', 'MCC', 'MCC CI 95%'], loc='lower right', fontsize=11)
plt.ylim(0.2,0.7)
plt.ylabel('Score')
plt.xticks(ticks=[1000, 10000], labels=['1000', '10000'], fontweight='bold')
plt.yticks(ticks=[0.2, 0.3, 0.4, 0.5, 0.6, 0.7], labels=['0.2', '0.3', '0.4', '0.5', '0.6', '0.7'], fontweight='bold')
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(1)
ax.tick_params(
    axis='both',
    which='major',
    direction='out',
    length=5,
    width=1,
    color='black',
    pad=5,
    bottom=True,  # Ensure ticks are drawn on the required edges
    left=True,    # Ensure ticks are drawn on the required edges
    top=False,    # Typically, ticks are not shown on the top and right
    right=False   # Color of the ticks (can be adjusted as needed)
)
ax.tick_params(
    axis='x', # Apply only to the X-axis
    which='minor',
    direction='out',
    length=2.5, # Minor ticks are usually shorter than major ticks
    width=0.7,  # Minor ticks are usually thinner than major ticks
    color='black', # Consistent or slightly lighter than major ticks
    bottom=True,
    top=False
)
plt.show()

### 3.2 using 5_class
- 5_class
- sp_mode: `s`
- feat: `tsfresh_raw`
- ag_train_quality: `medium_quality`
- hours: `0.2`

It may take ~40 minutes, or you can skip training and use our results in `data_efficiency_test_result.csv`

In [ ]:
# You can skip this cell and use our results in `data_efficiency_test_result.csv`

class_list = [
    'S1',
    'S2',
    'S3',
    'S4',
    'S5',
]
class_num = len(class_list)

ag_model_root = './AutogluonModels/'
sp_mode = 's'
feat = 'tsfresh_raw'
ag_train_quality = 'medium_quality'
hours = 0.2
label_col = 'class'
random_seed = 42

train_val_tsfresh_data = pd.read_csv(os.path.join(f'data/', f'{str(class_num)}_class', 'train_val_tsfresh_feat.csv'))
train_val_raw_data = pd.read_csv(os.path.join(f'data/', f'{str(class_num)}_class', 'train_val.csv'))
test_tsfresh_data = pd.read_csv(os.path.join(f'data/', f'{str(class_num)}_class', 'test_tsfresh_feat.csv'))
test_raw_data = pd.read_csv(os.path.join(f'data/', f'{str(class_num)}_class', 'test.csv'))
train_val_raw_data.drop(columns=['class'], inplace=True)
test_raw_data.drop(columns=['class'], inplace=True)
train_val_tsfresh_data.drop(columns=[c for c in train_val_tsfresh_data.columns if c.startswith('p')], inplace=True)
train_val_raw_data.drop(columns=[c for c in train_val_raw_data.columns if c.startswith('p')], inplace=True)
test_tsfresh_data.drop(columns=[c for c in test_tsfresh_data.columns if c.startswith('p')], inplace=True)
test_raw_data.drop(columns=[c for c in test_raw_data.columns if c.startswith('p')], inplace=True)

train_val_merge_data = pd.merge(train_val_tsfresh_data, train_val_raw_data, on='id')
test_merge_data = pd.merge(test_tsfresh_data, test_raw_data, on='id')

train_val_merge_data = train_val_merge_data.sample(frac=1, random_state=random_seed).reset_index(drop=True)
test_merge_data = test_merge_data.sample(frac=1, random_state=random_seed).reset_index(drop=True)
train_val_merge_data.replace([np.inf, -np.inf], np.nan, inplace=True)
test_merge_data.replace([np.inf, -np.inf], np.nan, inplace=True)

train_val_merge_data.head()

In [ ]:
# You can skip this cell and use our results in `data_efficiency_test_result.csv`

import warnings

warnings.filterwarnings("ignore", category=UserWarning)

all_data = pd.concat([train_val_merge_data, test_merge_data], ignore_index=True)
data_size_list = np.array([500, 1000, 1500, 2000, 3000, 4000, 5000, 6000, 8000, 10000])
print(data_size_list)
test_result_df = pd.DataFrame(columns=['Data_Size', 'Accuracy', 'MCC'])
repeat_num = 5
pbar = tqdm(total=len(data_size_list) * repeat_num)
for data_size in data_size_list:
    for repeat in range(repeat_num):
        try:
            sample_data = train_test_split(all_data, train_size=data_size, shuffle=True, stratify=all_data['old_class'], )[0]
        except:
            sample_data = train_test_split(all_data, train_size=data_size, shuffle=True, stratify=all_data['class'], )[0]
        try:
            train_val_merge_data_, test_merge_data_ = train_test_split(sample_data, test_size=0.1, shuffle=True, stratify=sample_data['old_class'], )
        except:
            train_val_merge_data_, test_merge_data_ = train_test_split(sample_data, test_size=0.1, shuffle=True, stratify=sample_data['class'], )
        # print(train_val_merge_data_.shape, test_merge_data_.shape)

        metric = 'accuracy'

        predictor = TabularPredictor(
            problem_type='multiclass',
            label=label_col,
            learner_kwargs={'ignored_columns': ['id', 'old_class']},
            eval_metric=metric,
            path=f'./AutogluonModels/{str(class_num)}_class/benchmark_data_size__{data_size}__{repeat}/{feat}_feat__{sp_mode}__{ag_train_quality}__{hours}h',
            verbosity=0,
        ).fit(
            train_data=train_val_merge_data_,
            num_gpus=1,
            num_cpus=16,
            presets=ag_train_quality,
            time_limit=60*60*hours,
        )
        test_result = predictor.evaluate(test_merge_data_)
        test_result_df.loc[len(test_result_df)] = [data_size, test_result['accuracy'], test_result['mcc']]

        pbar.update(1)
test_result_df.to_csv(f'./AutogluonModels/{str(class_num)}_class/data_efficiency_test_result.csv', index=False)

In [ ]:
# You can use our results in `data_efficiency_test_result.csv`

class_list = [
    'S1',
    'S2',
    'S3',
    'S4',
    'S5',
]
class_num = len(class_list)

test_result_df = pd.read_csv(f'./AutogluonModels/{str(class_num)}_class/data_efficiency_test_result.csv')
print(f'load {str(class_num)}_class data_efficiency_test_result.csv')
test_result_df

In [ ]:
plt.figure(figsize=(4.5, 4), dpi=300)
sns.lineplot(data=test_result_df, x='Data_Size', y='Accuracy', marker='o', 
             errorbar=('ci', 95),
             err_style='band',)
sns.lineplot(data=test_result_df, x='Data_Size', y='MCC', marker='o', 
             errorbar=('ci', 95),
             err_style='band',)

# plt.fill_between
plt.xscale('log')
plt.legend(['Accuracy', 'Accuracy CI 95%', 'MCC', 'MCC CI 95%'], loc='lower right', fontsize=11)
plt.ylim(0.4,0.85)
plt.ylabel('Score')
plt.xlabel('Data Size')
plt.xticks(ticks=[1000, 10000], labels=['1000', '10000'], fontweight='bold')
plt.yticks(ticks=[0.4, 0.5, 0.6, 0.7, 0.8], labels=['0.4', '0.5', '0.6', '0.7', '0.8'], fontweight='bold')
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(1)
ax.tick_params(
    axis='both',
    which='major',
    direction='out',
    length=5,
    width=1,
    color='black',
    pad=5,
    bottom=True,  # Ensure ticks are drawn on the required edges
    left=True,    # Ensure ticks are drawn on the required edges
    top=False,    # Typically, ticks are not shown on the top and right
    right=False   # Color of the ticks (can be adjusted as needed)
)
ax.tick_params(
    axis='x', # Apply only to the X-axis
    which='minor',
    direction='out',
    length=2.5, # Minor ticks are usually shorter than major ticks
    width=0.7,  # Minor ticks are usually thinner than major ticks
    color='black', # Consistent color with major ticks or slightly lighter
    bottom=True,
    top=False
)
plt.show()

## 4. Benchmark with feature (Table S3)

### 4.1 using 4_class
features:
1. tsfresh + raw (we have provided on Zenodo)
2. raw (you might need to train it yourself)
3. tsfresh (you might need to train it yourself)

You can load the results in `feature_test_result.csv` because the size of other models is too large to upload.

In [ ]:
# You can skip this cell and use our results in `feature_test_result.csv`

class_list = [
    'S1',
    'S2',
    'S3',
    'S4',
]
class_num = len(class_list)
ag_model_root = './AutogluonModels/'
sp_mode = 's'
ag_train_quality = 'best_quality'
hours = 3.0
label_col = 'class'
random_seed = 42

# load test data
test_tsfresh_data = pd.read_csv(os.path.join(f'data/', f'{str(class_num)}_class', 'test_tsfresh_feat.csv'))
test_raw_data = pd.read_csv(os.path.join(f'data/', f'{str(class_num)}_class', 'test.csv'))
if sp_mode == 's':
    test_tsfresh_data.drop(columns=[c for c in test_tsfresh_data.columns if c.startswith('p')], inplace=True)
    test_raw_data.drop(columns=[c for c in test_raw_data.columns if c.startswith('p')], inplace=True)
elif sp_mode == 'p':
    test_tsfresh_data.drop(columns=[c for c in test_tsfresh_data.columns if c.startswith('s')], inplace=True)
    test_raw_data.drop(columns=[c for c in test_raw_data.columns if c.startswith('s')], inplace=True)
test_raw_data.drop(columns=['class'], inplace=True)
test_merge_data = pd.merge(test_tsfresh_data, test_raw_data, on='id')
test_merge_data.replace([np.inf, -np.inf], np.nan, inplace=True)
y_test = test_merge_data[label_col]
test_merge_data_nolabel = test_merge_data.drop(columns=[label_col])  # delete label column
test_merge_data_nolabel.head()

df_results = pd.DataFrame(columns=['Feature', 'Metrics', 'Score'])
for feat in ['tsfresh_raw', 'raw', 'tsfresh']:
    model_root = os.path.join(ag_model_root, f'{str(class_num)}_class', f'{feat}_feat__{sp_mode}__{ag_train_quality}__{hours}h')
    predictor = TabularPredictor.load(model_root)
    
    # predict on test data
    y_pred = predictor.predict(test_merge_data_nolabel)
    y_proba = predictor.predict_proba(test_merge_data_nolabel)

    accuracy = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba, multi_class="ovr")
    mcc = matthews_corrcoef(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted")
    precision = precision_score(y_test, y_pred, average="weighted")
    recall = recall_score(y_test, y_pred, average="weighted")
    top2_accuracy = top_k_accuracy_score(y_test, y_proba, k=2)
    top3_accuracy = top_k_accuracy_score(y_test, y_proba, k=3)

    df_results.loc[len(df_results)] = [feat, 'Accuracy', accuracy]
    df_results.loc[len(df_results)] = [feat, 'ROC_AUC', roc_auc]
    df_results.loc[len(df_results)] = [feat, 'MCC', mcc]
    df_results.loc[len(df_results)] = [feat, 'F1', f1]
    df_results.loc[len(df_results)] = [feat, 'Precision', precision]
    df_results.loc[len(df_results)] = [feat, 'Recall', recall]
    df_results.loc[len(df_results)] = [feat, 'Top2_Accuracy', top2_accuracy]
    df_results.loc[len(df_results)] = [feat, 'Top3_Accuracy', top3_accuracy]

df_results.to_csv(f'./AutogluonModels/{str(class_num)}_class/feature_test_result.csv', index=False)
df_results

In [ ]:
# You can use our results in `feature_test_result.csv` to plot the bar plot

class_list = [
    'S1',
    'S2',
    'S3',
    'S4',
]
class_num = len(class_list)

df_results = pd.read_csv(f'./AutogluonModels/{str(class_num)}_class/feature_test_result.csv')

fig = plt.figure(figsize=(6, 6))
fig = sns.barplot(data=df_results, x='Score', y='Metrics', hue='Feature', legend=True)
custom_handles = [
    plt.Line2D([0], [0], marker='o', markersize=10, linestyle='none', label='tsfresh_raw'),
    plt.Line2D([0], [0], marker='o', markersize=10, linestyle='none', label='raw'),
    plt.Line2D([0], [0], marker='o', markersize=10, linestyle='none', label='tsfresh'),
]

fig.set_xlim(0.7, 1)

### 4.2 using 5_class
features:
1. tsfresh + raw (we have provided on Zenodo)
2. raw (you might need to train it yourself)
3. tsfresh (you might need to train it yourself)

You can load the results in `feature_test_result.csv` because the size of other models is too large to upload.

In [ ]:
# You can skip this cell and use our results in `feature_test_result.csv`

class_list = [
    'S1',
    'S2',
    'S3',
    'S4',
    'S5',
]
class_num = len(class_list)
ag_model_root = './AutogluonModels/'
sp_mode = 's'
ag_train_quality = 'best_quality'
hours = 3.0
label_col = 'class'
random_seed = 42

# load test data
test_tsfresh_data = pd.read_csv(os.path.join(f'data/', f'{str(class_num)}_class', 'test_tsfresh_feat.csv'))
test_raw_data = pd.read_csv(os.path.join(f'data/', f'{str(class_num)}_class', 'test.csv'))
if sp_mode == 's':
    test_tsfresh_data.drop(columns=[c for c in test_tsfresh_data.columns if c.startswith('p')], inplace=True)
    test_raw_data.drop(columns=[c for c in test_raw_data.columns if c.startswith('p')], inplace=True)
elif sp_mode == 'p':
    test_tsfresh_data.drop(columns=[c for c in test_tsfresh_data.columns if c.startswith('s')], inplace=True)
    test_raw_data.drop(columns=[c for c in test_raw_data.columns if c.startswith('s')], inplace=True)

test_raw_data.drop(columns=['class'], inplace=True)
test_merge_data = pd.merge(test_tsfresh_data, test_raw_data, on='id')
test_merge_data.replace([np.inf, -np.inf], np.nan, inplace=True)
y_test = test_merge_data[label_col]
test_merge_data_nolabel = test_merge_data.drop(columns=[label_col])  # delete label column
test_merge_data_nolabel.head()

df_results = pd.DataFrame(columns=['Feature', 'Metrics', 'Score'])
for feat in ['tsfresh_raw', 'raw', 'tsfresh']:
    model_root = os.path.join(ag_model_root, f'{str(class_num)}_class', f'{feat}_feat__{sp_mode}__{ag_train_quality}__{hours}h')
    predictor = TabularPredictor.load(model_root)
    
    # predict on test data
    y_pred = predictor.predict(test_merge_data_nolabel)
    y_proba = predictor.predict_proba(test_merge_data_nolabel)

    accuracy = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba, multi_class="ovr")
    mcc = matthews_corrcoef(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted")
    precision = precision_score(y_test, y_pred, average="weighted")
    recall = recall_score(y_test, y_pred, average="weighted")
    top2_accuracy = top_k_accuracy_score(y_test, y_proba, k=2)
    top3_accuracy = top_k_accuracy_score(y_test, y_proba, k=3)

    df_results.loc[len(df_results)] = [feat, 'Accuracy', accuracy]
    df_results.loc[len(df_results)] = [feat, 'ROC_AUC', roc_auc]
    df_results.loc[len(df_results)] = [feat, 'MCC', mcc]
    df_results.loc[len(df_results)] = [feat, 'F1', f1]
    df_results.loc[len(df_results)] = [feat, 'Precision', precision]
    df_results.loc[len(df_results)] = [feat, 'Recall', recall]
    df_results.loc[len(df_results)] = [feat, 'Top2_Accuracy', top2_accuracy]
    df_results.loc[len(df_results)] = [feat, 'Top3_Accuracy', top3_accuracy]

df_results.to_csv(f'./AutogluonModels/{str(class_num)}_class/feature_test_result.csv', index=False)
df_results

In [ ]:
# You can use our results in `feature_test_result.csv` to plot the bar plot

class_list = [
    'S1',
    'S2',
    'S3',
    'S4',
    'S5',
]
class_num = len(class_list)
df_results = pd.read_csv(f'./AutogluonModels/{str(class_num)}_class/feature_test_result.csv')

fig = plt.figure(figsize=(6, 6))
fig = sns.barplot(data=df_results, x='Score', y='Metrics', hue='Feature', legend=True)

custom_handles = [
    plt.Line2D([0], [0], marker='o', markersize=10, linestyle='none', label='tsfresh_raw'),
    plt.Line2D([0], [0], marker='o', markersize=10, linestyle='none', label='raw'),
    plt.Line2D([0], [0], marker='o', markersize=10, linestyle='none', label='tsfresh'),
]

fig.set_xlim(0.6, 1)

## 5. Benchmark with training time

### 5.1 using 4_class
time:
1. 1.0 hour (you might need to train it yourself)
2. 2.0 hours (you might need to train it yourself)
3. 3.0 hours (we have provided on Zenodo)

You can load the results in `time_test_result.csv` because the size of other models is too large to upload.

In [ ]:
# You can skip this cell and use our results in `time_test_result.csv`

class_list = [
    'S1',
    'S2',
    'S3',
    'S4',
]
class_num = len(class_list)
ag_model_root = './AutogluonModels/'
sp_mode = 's'
ag_train_quality = 'best_quality'
feat = 'tsfresh_raw'
label_col = 'class'
random_seed = 42

# load test data
test_tsfresh_data = pd.read_csv(os.path.join(f'data/', f'{str(class_num)}_class', 'test_tsfresh_feat.csv'))
test_raw_data = pd.read_csv(os.path.join(f'data/', f'{str(class_num)}_class', 'test.csv'))
if sp_mode == 's':
    test_tsfresh_data.drop(columns=[c for c in test_tsfresh_data.columns if c.startswith('p')], inplace=True)
    test_raw_data.drop(columns=[c for c in test_raw_data.columns if c.startswith('p')], inplace=True)
elif sp_mode == 'p':
    test_tsfresh_data.drop(columns=[c for c in test_tsfresh_data.columns if c.startswith('s')], inplace=True)
    test_raw_data.drop(columns=[c for c in test_raw_data.columns if c.startswith('s')], inplace=True)
test_raw_data.drop(columns=['class'], inplace=True)
test_merge_data = pd.merge(test_tsfresh_data, test_raw_data, on='id')
test_merge_data.replace([np.inf, -np.inf], np.nan, inplace=True)
y_test = test_merge_data[label_col]
test_merge_data_nolabel = test_merge_data.drop(columns=[label_col])  # delete label column
test_merge_data_nolabel.head()

df_results = pd.DataFrame(columns=['Hour', 'Metrics', 'Score'])
for hours in [1.0, 2.0, 3.0]:
    model_root = os.path.join(ag_model_root, f'{str(class_num)}_class', f'{feat}_feat__{sp_mode}__{ag_train_quality}__{str(hours)}h')
    predictor = TabularPredictor.load(model_root)
    
    # predict on test data
    y_pred = predictor.predict(test_merge_data_nolabel)
    y_proba = predictor.predict_proba(test_merge_data_nolabel)

    accuracy = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba, multi_class="ovr")
    mcc = matthews_corrcoef(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted")
    precision = precision_score(y_test, y_pred, average="weighted")
    recall = recall_score(y_test, y_pred, average="weighted")
    top2_accuracy = top_k_accuracy_score(y_test, y_proba, k=2)
    top3_accuracy = top_k_accuracy_score(y_test, y_proba, k=3)

    df_results.loc[len(df_results)] = [hours, 'Accuracy', accuracy]
    df_results.loc[len(df_results)] = [hours, 'ROC_AUC', roc_auc]
    df_results.loc[len(df_results)] = [hours, 'MCC', mcc]
    df_results.loc[len(df_results)] = [hours, 'F1', f1]
    df_results.loc[len(df_results)] = [hours, 'Precision', precision]
    df_results.loc[len(df_results)] = [hours, 'Recall', recall]
    df_results.loc[len(df_results)] = [hours, 'Top2_Accuracy', top2_accuracy]
    df_results.loc[len(df_results)] = [hours, 'Top3_Accuracy', top3_accuracy]

df_results.to_csv(f'./AutogluonModels/{str(class_num)}_class/time_test_result.csv', index=False)
df_results

In [ ]:
# You can use our results in `time_test_result.csv` to plot the bar plot

class_list = [
    'S1',
    'S2',
    'S3',
    'S4',
]
class_num = len(class_list)
df_results = pd.read_csv(f'./AutogluonModels/{str(class_num)}_class/time_test_result.csv')  

fig = plt.figure(figsize=(6, 6))
fig = sns.barplot(data=df_results, x='Score', y='Metrics', hue='Hour', legend=True)

# custom_handles = [
#     plt.Line2D([0], [0], marker='o', markersize=10, linestyle='none', label='tsfresh_raw'),
#     plt.Line2D([0], [0], marker='o', markersize=10, linestyle='none', label='raw'),
#     plt.Line2D([0], [0], marker='o', markersize=10, linestyle='none', label='tsfresh'),
# ]

fig.set_xlim(0.7, 1)

### 5.2 using 5_class
time:
1. 1.0 hour (you might need to train it yourself)
2. 2.0 hours (you might need to train it yourself)
3. 3.0 hours (we have provided on Zenodo)

You can load the results in `time_test_result.csv` because the size of other models is too large to upload.

In [ ]:
# You can skip this cell and use our results in `time_test_result.csv`

class_list = [
    'S1',
    'S2',
    'S3',
    'S4',
    'S5',
]
class_num = len(class_list)
ag_model_root = './AutogluonModels/'
sp_mode = 's'
ag_train_quality = 'best_quality'
feat = 'tsfresh_raw'
label_col = 'class'
random_seed = 42

# load test data
test_tsfresh_data = pd.read_csv(os.path.join(f'data/', f'{str(class_num)}_class', 'test_tsfresh_feat.csv'))
test_raw_data = pd.read_csv(os.path.join(f'data/', f'{str(class_num)}_class', 'test.csv'))
if sp_mode == 's':
    test_tsfresh_data.drop(columns=[c for c in test_tsfresh_data.columns if c.startswith('p')], inplace=True)
    test_raw_data.drop(columns=[c for c in test_raw_data.columns if c.startswith('p')], inplace=True)
elif sp_mode == 'p':
    test_tsfresh_data.drop(columns=[c for c in test_tsfresh_data.columns if c.startswith('s')], inplace=True)
    test_raw_data.drop(columns=[c for c in test_raw_data.columns if c.startswith('s')], inplace=True)
test_raw_data.drop(columns=['class'], inplace=True)
test_merge_data = pd.merge(test_tsfresh_data, test_raw_data, on='id')
test_merge_data.replace([np.inf, -np.inf], np.nan, inplace=True)
y_test = test_merge_data[label_col]
test_merge_data_nolabel = test_merge_data.drop(columns=[label_col])  # delete label column
test_merge_data_nolabel.head()

df_results = pd.DataFrame(columns=['Hour', 'Metrics', 'Score'])
for hours in [1.0, 2.0, 3.0]:
    model_root = os.path.join(ag_model_root, f'{str(class_num)}_class', f'{feat}_feat__{sp_mode}__{ag_train_quality}__{str(hours)}h')
    predictor = TabularPredictor.load(model_root)
    
    # predict on test data
    y_pred = predictor.predict(test_merge_data_nolabel)
    y_proba = predictor.predict_proba(test_merge_data_nolabel)

    accuracy = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba, multi_class="ovr")
    mcc = matthews_corrcoef(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted")
    precision = precision_score(y_test, y_pred, average="weighted")
    recall = recall_score(y_test, y_pred, average="weighted")
    top2_accuracy = top_k_accuracy_score(y_test, y_proba, k=2)
    top3_accuracy = top_k_accuracy_score(y_test, y_proba, k=3)

    df_results.loc[len(df_results)] = [hours, 'Accuracy', accuracy]
    df_results.loc[len(df_results)] = [hours, 'ROC_AUC', roc_auc]
    df_results.loc[len(df_results)] = [hours, 'MCC', mcc]
    df_results.loc[len(df_results)] = [hours, 'F1', f1]
    df_results.loc[len(df_results)] = [hours, 'Precision', precision]
    df_results.loc[len(df_results)] = [hours, 'Recall', recall]
    df_results.loc[len(df_results)] = [hours, 'Top2_Accuracy', top2_accuracy]
    df_results.loc[len(df_results)] = [hours, 'Top3_Accuracy', top3_accuracy]

df_results.to_csv(f'./AutogluonModels/{str(class_num)}_class/time_test_result.csv', index=False)
df_results

In [ ]:
# You can use our results in `time_test_result.csv` to plot the bar plot

class_list = [
    'S1',
    'S2',
    'S3',
    'S4',
    'S5',
]
class_num = len(class_list)
df_results = pd.read_csv(f'./AutogluonModels/{str(class_num)}_class/time_test_result.csv')  

fig = plt.figure(figsize=(6, 6))
fig = sns.barplot(data=df_results, x='Score', y='Metrics', hue='Hour', legend=True)

# custom_handles = [
#     plt.Line2D([0], [0], marker='o', markersize=10, linestyle='none', label='tsfresh_raw'),
#     plt.Line2D([0], [0], marker='o', markersize=10, linestyle='none', label='raw'),
#     plt.Line2D([0], [0], marker='o', markersize=10, linestyle='none', label='tsfresh'),
# ]

fig.set_xlim(0.7, 1)

## 6. Feature Importance Analysis

In [ ]:
from autogluon.tabular import TabularPredictor
import pandas as pd
import numpy as np
import os

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="white")
sns.set_context("talk")
plt.rcParams['font.family'] = 'Arial'

In [ ]:
class_list = [
    'S1',
    'S2',
    'S3',
    'S4',
    'S5',
]
class_num = len(class_list)

ag_model_root = './AutogluonModels/'
sp_mode = 's'
feat = 'tsfresh_raw'
ag_train_quality = 'best_quality'
hours = 3.0
label_col = 'class'
random_seed = 42

# load test data
test_tsfresh_data = pd.read_csv(os.path.join(f'data/', f'{str(class_num)}_class', 'test_tsfresh_feat.csv'))
test_raw_data = pd.read_csv(os.path.join(f'data/', f'{str(class_num)}_class', 'test.csv'))
if sp_mode == 's':
    test_tsfresh_data.drop(columns=[c for c in test_tsfresh_data.columns if c.startswith('p')], inplace=True)
    test_raw_data.drop(columns=[c for c in test_raw_data.columns if c.startswith('p')], inplace=True)
elif sp_mode == 'p':
    test_tsfresh_data.drop(columns=[c for c in test_tsfresh_data.columns if c.startswith('s')], inplace=True)
    test_raw_data.drop(columns=[c for c in test_raw_data.columns if c.startswith('s')], inplace=True)
if feat == 'tsfresh':
    test_merge_data = test_tsfresh_data
elif feat == 'raw':
    test_merge_data = test_raw_data
elif feat == 'tsfresh_raw':
    test_raw_data.drop(columns=['class'], inplace=True)
    test_merge_data = pd.merge(test_tsfresh_data, test_raw_data, on='id')
test_merge_data.replace([np.inf, -np.inf], np.nan, inplace=True)
y_test = test_merge_data[label_col]
test_merge_data_nolabel = test_merge_data.drop(columns=[label_col])  # delete label column

# load model
model_root = os.path.join(ag_model_root, f'{str(class_num)}_class', f'{feat}_feat__{sp_mode}__{ag_train_quality}__{hours}h')
predictor = TabularPredictor.load(model_root)
predictor.leaderboard()

In [ ]:
# a quick version (10 mins)
df_feature_importance = predictor.feature_importance(
    data = test_merge_data,
    subsample_size = 1000,
    time_limit = 10*60,
    num_shuffle_sets = 1,
)
df_feature_importance.to_csv(f'feature_importance_{len(class_list)}class_1000subsample_1shuffle_600s.csv')

df_feature_importance

In [ ]:
# this may take 20 hours
df_feature_importance_best = predictor.feature_importance(
    data = test_merge_data,
    subsample_size = 5000,
    num_shuffle_sets = 10,
)
df_feature_importance_best.to_csv(f'feature_importance_{len(class_list)}class_5000subsample_10shuffle.csv')

df_feature_importance_best